# Day 4 — LLM Observability with Langfuse

---

You have a deployed app. Users are hitting it. Some queries are slow, some cost too much, some produce weird answers. **How do you know?**

Regular web observability (Datadog, New Relic, Prometheus) doesn't understand LLM concepts: prompts, tokens, tool calls, chains. **LLM observability** tools do.

**Langfuse** is the open-source leader. Free hosted tier at https://langfuse.com. Alternatives: **Arize Phoenix**, **LangSmith** (from the LangChain team). All are similar; we pick Langfuse for the open-source + free-tier combo.


## 1. What you get from Langfuse

- Every LLM call is a **trace** with prompt, completion, latency, token counts, cost, and any tool calls
- Group traces by user, session, feature flag
- Slice cost by model / user / endpoint
- Score outputs manually or with an evaluator (like RAGAS or LLM-as-judge)
- All queryable in a web UI + API


## 2. Setup


In [ ]:
!pip install langfuse together python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Get keys at https://cloud.langfuse.com (free)
assert os.getenv("LANGFUSE_PUBLIC_KEY"), "Set LANGFUSE_PUBLIC_KEY"
assert os.getenv("LANGFUSE_SECRET_KEY"), "Set LANGFUSE_SECRET_KEY"

from langfuse import Langfuse
lf = Langfuse()


## 3. Trace a single LLM call


In [ ]:
from together import Together
llm = Together()

@lf.observe(as_type="generation", name="hello-call")
def call_llm(prompt: str) -> str:
    resp = llm.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    return resp.choices[0].message.content

print(call_llm("Say hi in 5 words."))


Open your Langfuse dashboard → **Traces**. You'll see one trace with:
- Prompt, completion
- Latency
- Model name
- (Token counts auto-populate if you use their Together integration)

You added *one decorator* and got a full observability trail. That's the pitch.


## 4. Trace a multi-step chain

For a RAG or agent flow, wrap each step:


In [ ]:
@lf.observe(name="retrieve")
def retrieve(question: str) -> list[str]:
    # ... vector search ...
    return ["chunk1", "chunk2"]

@lf.observe(name="generate")
def generate(question: str, chunks: list[str]) -> str:
    prompt = f"Context: {chunks}\n\nQuestion: {question}"
    return call_llm(prompt)

@lf.observe(name="rag")
def rag(question: str) -> str:
    chunks = retrieve(question)
    return generate(question, chunks)

print(rag("What is the capital of France?"))


In the dashboard, the top-level `rag` trace has `retrieve` and `generate` as **child spans**. You see exactly which step is slow, which step uses tokens, and which step failed if there was an error.

This is the **flame graph** view for LLM apps. Priceless when a user reports "the bot is slow" and you need to know why.


## 5. Add user + session context

Tag each trace with who and what session — makes debugging trivial.


In [ ]:
with lf.start_as_current_span(name="user-session") as span:
    span.update(user_id="alice", session_id="chat-42")
    answer = rag("Who founded AcmeCloud?")


Now in the dashboard you can filter to `user_id=alice` and see every trace from her session.


## 6. Score outputs — human or auto

Add a `/feedback` endpoint that lets users thumb up/down. Send the score to Langfuse:


In [ ]:
# lf.score(trace_id="...", name="user_thumbs", value=1)   # or 0 for thumbs-down

# For LLM-as-judge scoring, run periodically:
# from langfuse.decorators import observe
# ...call a judge model and lf.score(...)


Over time you'll see quality trend. Regressions after a prompt change stand out immediately.


## 7. What to actually watch

Dashboards to set up in your first week:

1. **p50 / p95 latency** per endpoint
2. **Cost per day** (input + output tokens × price)
3. **Error rate** per LLM call
4. **Top 10 slow traces** — a click-through for triage
5. **User feedback score** trend

If any of these degrade after a deploy, roll back. This is the ROI of observability.


## 8. Alternatives to know

- **Arize Phoenix** — free, self-hostable, great for RAG debugging
- **LangSmith** — from LangChain team, deep integration with LangGraph
- **Datadog LLM Observability** — if your company already uses Datadog
- **PostHog LLM analytics** — good if you already use PostHog for product analytics

Pick one based on your team's existing stack. **The important thing is you have one at all** — running an LLM app blind is malpractice.


## Recap

- **LLM observability** = tracing built for prompts, tokens, tool calls, chains.
- **Langfuse** — free open-source, generous cloud tier. One `@observe` decorator per function.
- Wrap **every step** of your RAG / agent flow for a flame-graph view.
- Tag traces with **user_id + session_id** for real debugging power.
- Score outputs (user thumbs + LLM-as-judge) to catch regressions.
- **Next class:** cost tracking and A/B testing.
